# Exploration 9 — Train Stage 2 (FAST default)
Uses only saved preprocessing artifacts. Resume/skip is automatic per arm × seed × fold.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
from pathlib import Path
import os

PUMA_ROOT = Path('/content/drive/MyDrive/Research/PUMA')
PROJECT_ROOT = PUMA_ROOT / 'Exploration_9_Natural_Transfer_Validation'
PREPROCESSED_ROOT = PROJECT_ROOT / 'PREPROCESSED'
TRAINING_ROOT = PROJECT_ROOT / 'TRAINING_RESULTS'
LOCAL_INPUT = Path('/content/e9_training_inputs')
LOCAL_RUNTIME = Path('/content/e9_runtime')
CPU_CORES = os.cpu_count() or 1
EXECUTION_MODE = 'fast'
CONTEXT_SCALE_INIT = 0.0
CONTEXT_INIT_PROVENANCE = 'reconstructed_zero_residual_default'
BOOTSTRAP_REPS = 5000

for path in (PROJECT_ROOT, PREPROCESSED_ROOT):
    if not path.exists():
        raise FileNotFoundError(path)
preprocess_status = PREPROCESSED_ROOT / 'PREPROCESS_STATUS.json'
if not preprocess_status.is_file():
    raise FileNotFoundError(preprocess_status)
import json
if json.loads(preprocess_status.read_text()).get('status') != 'PREPROCESS_COMPLETE':
    raise RuntimeError(f'Preprocessing is not complete: {preprocess_status}')
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
print('Execution mode:', EXECUTION_MODE.upper())
print('CPU cores:', CPU_CORES)
print('Project:', PROJECT_ROOT)


In [ ]:
import os, sys, subprocess, importlib.util
os.chdir(PROJECT_ROOT)

# Check every dependency imported by the Stage-2 training path.
# PIL is provided by Pillow; scipy is imported by tier_a_extract.
required_modules = ('numpy', 'torch', 'sklearn', 'scipy', 'PIL')
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    print('[DEPS] Missing', missing, '-> installing requirements.txt')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    print('[DEPS] Training dependencies already available -> SKIP pip install')


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
torch.set_float32_matmul_precision('highest')
torch.backends.cuda.matmul.allow_tf32 = False
print('GPU:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f'VRAM: {total/1024**3:.1f} GB total, {free/1024**3:.1f} GB free')


In [ ]:
# Copy the tiny runtime code to local SSD so one run uses one consistent code snapshot.
# This avoids Google Drive/FUSE import latency and stale __pycache__ issues.
import shutil

if LOCAL_RUNTIME.exists():
    shutil.rmtree(LOCAL_RUNTIME)

def _ignore_runtime_clutter(path, names):
    return [n for n in names if n in {'__pycache__', '.pytest_cache'} or n.endswith('.pyc')]

shutil.copytree(
    PROJECT_ROOT / 'CODE',
    LOCAL_RUNTIME / 'CODE',
    ignore=_ignore_runtime_clutter,
)
print('[CODE] Fresh local runtime snapshot:', LOCAL_RUNTIME / 'CODE')


In [ ]:
# Stage only the artifacts required by Stage-2 training from Drive -> local SSD.
# A metadata fingerprint makes reuse safe; if it changes, recopy everything so
# same-size stale .npy files can never be accepted accidentally.
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib, shutil, time

REQUIRED_PREPROCESSED = (
    'prepared/dataset_summary.json',
    'prepared/e9_natural_train_roster.csv',
    'prepared/tier_a.json',
    'prepared/tier_a.npy',
    'prepared/tier_a.uids.txt',
    'features/gaussian96.json',
    'features/gaussian96.npy',
    'features/gaussian96.uids.txt',
    'features/gaussian192.json',
    'features/gaussian192.npy',
    'features/gaussian192.uids.txt',
)

def source_fingerprint(root: Path) -> str:
    # These tiny metadata files already contain the manifest/UID/payload SHA256s.
    h = hashlib.sha256()
    for rel in (
        'prepared/dataset_summary.json',
        'prepared/tier_a.json',
        'features/gaussian96.json',
        'features/gaussian192.json',
    ):
        p = root / rel
        if not p.is_file():
            raise FileNotFoundError(p)
        h.update(rel.encode('utf-8'))
        h.update(p.read_bytes())
    return h.hexdigest()

fingerprint = source_fingerprint(PREPROCESSED_ROOT)
marker = LOCAL_INPUT / '.source_fingerprint'
source_files = [(PREPROCESSED_ROOT / rel, rel) for rel in REQUIRED_PREPROCESSED]
for src, _ in source_files:
    if not src.is_file():
        raise FileNotFoundError(src)

local_valid = (
    marker.is_file()
    and marker.read_text().strip() == fingerprint
    and all(
        (LOCAL_INPUT / rel).is_file()
        and (LOCAL_INPUT / rel).stat().st_size == src.stat().st_size
        for src, rel in source_files
    )
)

if local_valid:
    print('[STAGE] Local preprocessing cache matches Drive -> SKIP COPY')
else:
    # Fingerprint mismatch means source content changed. Recopy every required file,
    # not just size-mismatched files, because a changed .npy normally has the same size.
    if LOCAL_INPUT.exists():
        shutil.rmtree(LOCAL_INPUT)
    LOCAL_INPUT.mkdir(parents=True, exist_ok=True)
    jobs = []
    for src, rel in source_files:
        dst = LOCAL_INPUT / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        jobs.append((src, dst))
    total_bytes = sum(src.stat().st_size for src, _ in jobs)
    print(f'[STAGE] Copying {len(jobs)} required files ({total_bytes/1024**3:.2f} GB) with parallel Drive reads...')
    t0 = time.perf_counter()
    workers = min(4, CPU_CORES, max(1, len(jobs)))

    def copy_one(pair):
        src, dst = pair
        shutil.copyfile(src, dst)
        return src.name, src.stat().st_size

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(copy_one, pair) for pair in jobs]
        copied = 0
        for future in as_completed(futures):
            name, size = future.result()
            copied += size
            print(f'[STAGE] {name} done | {copied/1024**3:.2f}/{total_bytes/1024**3:.2f} GB', flush=True)
    marker.write_text(fingerprint)
    print(f'[STAGE] Complete in {(time.perf_counter()-t0):.1f}s')

PREP_DIR = LOCAL_INPUT / 'prepared'
FEATURE_DIR = LOCAL_INPUT / 'features'


In [ ]:
# FAST is the default. Stream the child process live so any real error is visible.
import subprocess, os, sys, shutil
from pathlib import Path

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONPATH'] = str(LOCAL_RUNTIME) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')

# Preflight critical imports from the exact local code snapshot that will train.
preflight = [
    sys.executable, '-u', '-c',
    (
        "import CODE.CLI.train; "
        "import CODE.TRAINING.run_transfer; "
        "import CODE.TRAINING.trainer; "
        "import CODE.TRAINING.loop; "
        "import CODE.MODELS.anchor; "
        "import CODE.MODELS.anchor_context_c; "
        "print('[PREFLIGHT] Runtime imports OK', flush=True)"
    ),
]
subprocess.run(preflight, cwd=LOCAL_RUNTIME, env=env, check=True)

cmd = [
    sys.executable, '-u', '-m', 'CODE.CLI.train',
    '--manifest', str(PREP_DIR / 'e9_natural_train_roster.csv'),
    '--feature-dir', str(FEATURE_DIR),
    '--tier-a-dir', str(PREP_DIR),
    '--output-dir', str(TRAINING_ROOT),
    '--device', 'cuda',
    '--cpu-threads', '0',
    '--bootstrap-reps', str(BOOTSTRAP_REPS),
    '--context-scale-init', str(CONTEXT_SCALE_INIT),
    '--context-init-provenance', CONTEXT_INIT_PROVENANCE,
    '--allow-exploratory-provenance',
    '--execution-mode', EXECUTION_MODE,
]

print('[LAUNCH]', ' '.join(cmd), flush=True)

drive_log_path = TRAINING_ROOT / 'train_console.log'
local_log_path = Path('/content/e9_train_console_session.log')

# Write the high-frequency console stream to local SSD, not Google Drive/FUSE.
# Sync it to Drive once when the child exits (success or failure).
with local_log_path.open('w', encoding='utf-8', buffering=1) as log:
    log.write('=== TRAIN SESSION ===\n')
    proc = subprocess.Popen(
        cmd,
        cwd=LOCAL_RUNTIME,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    return_code = proc.wait()

# One sequential Drive append instead of thousands of tiny FUSE writes.
with drive_log_path.open('a', encoding='utf-8') as dst, local_log_path.open('r', encoding='utf-8') as src:
    dst.write('\n\n=== NEW TRAIN LAUNCH ===\n')
    shutil.copyfileobj(src, dst, length=1024*1024)

if return_code != 0:
    raise RuntimeError(
        f'Training process failed with exit code {return_code}. '
        f'Full child-process log: {drive_log_path}'
    )

print('[DONE] Training command finished successfully.', flush=True)


In [ ]:
import json
summary_path = TRAINING_ROOT / 'NATURAL_TRANSFER_SUMMARY.json'
if summary_path.is_file():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))
else:
    print('Training is resumable; summary will appear after all fits finish.')
